# 📚 Content-Based Filtering (CBF) — Hệ Thống Gợi Ý Điểm Rèn Luyện CTU

## Mục tiêu
Xây dựng **Content-Based Filtering** cho hệ thống gợi ý hoạt động rèn luyện CTU Infinity.

## Công thức (theo `recommendation_system_analysis.md`)
```
Score_CBF(s, e) =
  w1 × deficit_score(s, e.criteriaId)   # w1 = 0.50
+ w2 × preference_sim(s, e)             # w2 = 0.25
+ w3 × not_attended(s, e)               # w3 = 0.10
+ w4 × upcoming_bonus(e)               # w4 = 0.10

preference_sim(s, e) =
  α × [cat_sim + organizer_sim + score_match]   # Implicit (lịch sử)
+ β × subscription_match(s, e)                  # Explicit (subscription)
```

## Mapping với DB thật (CTU Infinity Backend)
| Bảng DB | Entity | Cột dùng |
|---------|--------|----------|
| `events` | `Event` | `eventId`, `eventName`, `criteriaId`, `score`, `organizerId`, `startDate`, `endDate`, `description`, `status` |
| `event_category` | `EventCategory` | `categoryId`, `categoryName`, `slug` |
| `event_categories_mapping` | — | join table `events ↔ event_category` |
| `criterias` | `Criteria` | `criteriaId`, `criteriaCode`, `criteriaName`, `maxScore`, `parentId` |
| `organizers` | `Organizer` | `organizerId`, `organizerName` |
| `event_registrations` | `EventRegistration` | `studentId`, `eventId`, `status` (REGISTERED/ATTENDED/ABSENT/CANCELLED) |
| `student_scores` | `StudentScore` | `studentId`, `eventId`, `criteriaId`, `scoreValue` — **1 record/lần cộng** |
| `subscriptions` | `Subscription` | `studentId` |
| `subscription_categories` | — | join: `subscriptionId ↔ categoryId` |
| `subscription_criteria` | — | join: `subscriptionId ↔ criteriaId` |

> **Lưu ý quan trọng:** `student_scores` lưu từng lần cộng điểm riêng (mỗi attendance = 1 record).  
> Tổng điểm theo tiêu chí = `SUM(scoreValue) WHERE studentId=s AND criteriaId=c`.

---
## Cell 1: Import Thư Viện

In [2]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random
import uuid
import warnings
warnings.filterwarnings('ignore')

from sklearn.metrics.pairwise import cosine_similarity

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 60)
pd.set_option('display.float_format', '{:.4f}'.format)

random.seed(42)
np.random.seed(42)

print('✅ Import thư viện thành công!')

✅ Import thư viện thành công!


---
## Cell 2: Fake Dataset — Phản Ánh Đúng DB Thật

Toàn bộ bảng được mô phỏng theo đúng entity của CTU Infinity Backend.
IDs dùng format UUID (như DB thật). `student_scores` có **1 record mỗi lần cộng điểm**.

In [3]:
# ============================================================
# organizers (Organizer entity: organizerId, organizerName)
# ============================================================
df_organizers = pd.DataFrame([
    {'organizerId': 'org-btc-001',    'organizerName': 'Ban Tổ Chức Trường'},
    {'organizerId': 'org-doan-001',   'organizerName': 'Đoàn Thanh Niên CTU'},
    {'organizerId': 'org-hoisy-001',  'organizerName': 'Hội Sinh Viên CTU'},
    {'organizerId': 'org-khoa-001',   'organizerName': 'Khoa CNTT'},
    {'organizerId': 'org-ttht-001',   'organizerName': 'Trung Tâm Hỗ Trợ SV'},
])
organizer_ids = df_organizers['organizerId'].tolist()

# ============================================================
# criterias (Criteria entity: criteriaId, criteriaCode, criteriaName, maxScore, parentId, frameworkId)
# Gồm tiêu chí cha & con theo khung đánh giá DRL của CTU
# ============================================================
FRAMEWORK_ID = 'fw-ctu-drl-2024'
df_criterias = pd.DataFrame([
    # Tiêu chí cha
    {'criteriaId': 'crit-I',   'criteriaCode': 'I',    'criteriaName': 'Ý thức học tập',                   'maxScore': 20, 'parentId': None,      'frameworkId': FRAMEWORK_ID},
    {'criteriaId': 'crit-II',  'criteriaCode': 'II',   'criteriaName': 'Ý thức chấp hành nội quy',         'maxScore': 25, 'parentId': None,      'frameworkId': FRAMEWORK_ID},
    {'criteriaId': 'crit-III', 'criteriaCode': 'III',  'criteriaName': 'Ý thức & kết quả tham gia HĐNK',  'maxScore': 20, 'parentId': None,      'frameworkId': FRAMEWORK_ID},
    {'criteriaId': 'crit-IV',  'criteriaCode': 'IV',   'criteriaName': 'Ý thức công dân',                  'maxScore': 25, 'parentId': None,      'frameworkId': FRAMEWORK_ID},
    {'criteriaId': 'crit-V',   'criteriaCode': 'V',    'criteriaName': 'Công tác, hoạt động đoàn thể',    'maxScore': 10, 'parentId': None,      'frameworkId': FRAMEWORK_ID},
    # Tiêu chí con (có parentId, đây là tiêu chí gắn với sự kiện)
    {'criteriaId': 'crit-III-1', 'criteriaCode': 'III.1', 'criteriaName': 'Hoạt động học thuật',          'maxScore': 10, 'parentId': 'crit-III','frameworkId': FRAMEWORK_ID},
    {'criteriaId': 'crit-III-2', 'criteriaCode': 'III.2', 'criteriaName': 'Hoạt động văn thể mỹ',         'maxScore': 5,  'parentId': 'crit-III','frameworkId': FRAMEWORK_ID},
    {'criteriaId': 'crit-III-3', 'criteriaCode': 'III.3', 'criteriaName': 'Hoạt động tình nguyện',        'maxScore': 5,  'parentId': 'crit-III','frameworkId': FRAMEWORK_ID},
    {'criteriaId': 'crit-V-1',   'criteriaCode': 'V.1',   'criteriaName': 'Công tác Đoàn - Hội',          'maxScore': 5,  'parentId': 'crit-V',  'frameworkId': FRAMEWORK_ID},
    {'criteriaId': 'crit-V-2',   'criteriaCode': 'V.2',   'criteriaName': 'Công tác Đội - nhóm',          'maxScore': 5,  'parentId': 'crit-V',  'frameworkId': FRAMEWORK_ID},
])

# Tiêu chí lá (leaf) — gắn trực tiếp với sự kiện
leaf_criteria_ids = ['crit-III-1', 'crit-III-2', 'crit-III-3', 'crit-V-1', 'crit-V-2']
df_leaf_criteria = df_criterias[df_criterias['criteriaId'].isin(leaf_criteria_ids)]

# ============================================================
# event_category (EventCategory entity: categoryId, categoryName, slug)
# ============================================================
df_categories = pd.DataFrame([
    {'categoryId': 'cat-hoc-thuat', 'categoryName': 'Học thuật - Nghiên cứu',  'slug': 'hoc-thuat'},
    {'categoryId': 'cat-van-hoa',   'categoryName': 'Văn hoá - Nghệ thuật',    'slug': 'van-hoa-nghe-thuat'},
    {'categoryId': 'cat-the-thao',  'categoryName': 'Thể dục - Thể thao',      'slug': 'the-duc-the-thao'},
    {'categoryId': 'cat-tinh-nguyen','categoryName': 'Tình nguyện - Cộng đồng','slug': 'tinh-nguyen'},
    {'categoryId': 'cat-doan-hoi',  'categoryName': 'Đoàn - Hội sinh viên',    'slug': 'doan-hoi'},
    {'categoryId': 'cat-ky-nang',   'categoryName': 'Kỹ năng - Hướng nghiệp', 'slug': 'ky-nang'},
])
all_cat_ids = df_categories['categoryId'].tolist()

print('✅ Bảng tham chiếu đã tạo:')
print(f'   Organizers : {len(df_organizers)}')
print(f'   Criterias  : {len(df_criterias)} ({len(leaf_criteria_ids)} tiêu chí lá)')
print(f'   Categories : {len(df_categories)}')

✅ Bảng tham chiếu đã tạo:
   Organizers : 5
   Criterias  : 10 (5 tiêu chí lá)
   Categories : 6


In [5]:
# ============================================================
# events (Event entity)
# Trường: eventId(uuid), eventName, criteriaId, score, organizerId,
#         startDate, endDate, description, status='APPROVED'
# categories: list of categoryId (qua join table event_categories_mapping)
# ============================================================
criteria_category_map = {
    'crit-III-1': ['cat-hoc-thuat', 'cat-ky-nang'],
    'crit-III-2': ['cat-van-hoa', 'cat-the-thao'],
    'crit-III-3': ['cat-tinh-nguyen'],
    'crit-V-1'  : ['cat-doan-hoi'],
    'crit-V-2'  : ['cat-doan-hoi', 'cat-tinh-nguyen'],
}

event_templates = [
    # (eventName, criteriaId, score, description, organizerId)
    ('Hội thảo AI & Machine Learning',    'crit-III-1', 5,  'Seminar về AI, ML và ứng dụng thực tế',         'org-khoa-001'),
    ('Cuộc thi Lập trình Sinh viên CTU',  'crit-III-1', 8,  'Thi lập trình thuật toán cấp trường',           'org-khoa-001'),
    ('Seminar Khoa học Dữ liệu',          'crit-III-1', 4,  'Workshop phân tích dữ liệu với Python',         'org-khoa-001'),
    ('Workshop Kỹ năng Mềm',             'crit-III-1', 3,  'Rèn kỹ năng giao tiếp, làm việc nhóm',          'org-ttht-001'),
    ('Ngày hội Học thuật Khoa CNTT',     'crit-III-1', 5,  'Trình bày đề tài nghiên cứu sinh viên',         'org-khoa-001'),
    ('Liên hoan Văn nghệ Sinh viên',     'crit-III-2', 5,  'Biểu diễn văn nghệ các khoa',                   'org-btc-001'),
    ('Giải Cầu Lông Sinh viên',          'crit-III-2', 4,  'Giải thể thao cấp trường môn cầu lông',         'org-btc-001'),
    ('Đêm nhạc CTU Got Talent',          'crit-III-2', 6,  'Cuộc thi tài năng sinh viên',                   'org-hoisy-001'),
    ('Giải Bóng Đá Khoa',               'crit-III-2', 5,  'Giải bóng đá giao lưu khoa CNTT',               'org-khoa-001'),
    ('Triển lãm Mỹ thuật Sinh viên',    'crit-III-2', 3,  'Trưng bày tác phẩm mỹ thuật sinh viên',         'org-btc-001'),
    ('Hiến máu nhân đạo',               'crit-III-3', 10, 'Chiến dịch hiến máu tình nguyện',               'org-doan-001'),
    ('Mùa hè xanh Tình nguyện',         'crit-III-3', 8,  'Tình nguyện hỗ trợ cộng đồng vùng sâu',        'org-doan-001'),
    ('Nhặt rác bảo vệ môi trường',      'crit-III-3', 5,  'Dọn vệ sinh môi trường khuôn viên trường',      'org-hoisy-001'),
    ('Hỗ trợ đồng bào lũ lụt',         'crit-III-3', 10, 'Quyên góp, hỗ trợ vùng thiên tai',             'org-doan-001'),
    ('Trao học bổng trẻ em nghèo',      'crit-III-3', 7,  'Chương trình học bổng vùng khó khăn',           'org-hoisy-001'),
    ('Đại hội Đoàn Khoa CNTT',         'crit-V-1',   6,  'Đại hội đại biểu Đoàn TNCS khoa CNTT',         'org-doan-001'),
    ('Kết nạp Đoàn viên mới',          'crit-V-1',   5,  'Lễ kết nạp đoàn viên mới học kỳ 1',             'org-doan-001'),
    ('Hội nghị Ban Chấp hành Hội SV',  'crit-V-1',   4,  'Họp ban chấp hành hội sinh viên CTU',           'org-hoisy-001'),
    ('Chào tân sinh viên 2025-2026',   'crit-V-2',   5,  'Lễ đón tiếp tân sinh viên năm học mới',         'org-btc-001'),
    ('Ngày hội việc làm CTU',          'crit-V-2',   4,  'Kết nối sinh viên và doanh nghiệp tuyển dụng',  'org-ttht-001'),
]

now = datetime.now()
events_data = []
for i, (name, cid, score, desc, org_id) in enumerate(event_templates):
    cats = criteria_category_map.get(cid, ['cat-hoc-thuat'])
    days_offset = random.randint(-90, -5) if i < 14 else random.randint(3, 50)
    start = now + timedelta(days=days_offset)
    events_data.append({
        'eventId'     : f'evt-{i+1:03d}',          # UUID in production
        'eventName'   : name,
        'criteriaId'  : cid,
        'score'       : float(score),
        'organizerId' : org_id,
        'startDate'   : start,
        'endDate'     : start + timedelta(hours=random.randint(2, 6)),
        'description' : desc,
        'status'      : 'APPROVED',
        # categories = list of categoryId (join table event_categories_mapping)
        'categoryIds' : cats,
    })

df_events = pd.DataFrame(events_data)
df_events['is_upcoming'] = df_events['startDate'] > now

# score_normalized: chuẩn hóa score theo maxScore của criteria tương ứng
def get_max_score(cid):
    row = df_criterias[df_criterias['criteriaId']==cid]
    return float(row['maxScore'].values[0]) if len(row) > 0 else 10.0

df_events['score_normalized'] = df_events.apply(
    lambda r: r['score'] / get_max_score(r['criteriaId']), axis=1
)

print(f'📅 Sự kiện: {len(df_events)} tổng ({df_events["is_upcoming"].sum()} sắp tới, {(~df_events["is_upcoming"]).sum()} đã qua)')
print()
print(df_events[['eventId','eventName','criteriaId','score','score_normalized','organizerId','is_upcoming']].head(8).to_string(index=False))

📅 Sự kiện: 20 tổng (6 sắp tới, 14 đã qua)

eventId                        eventName criteriaId  score  score_normalized   organizerId  is_upcoming
evt-001   Hội thảo AI & Machine Learning crit-III-1 5.0000            0.5000  org-khoa-001        False
evt-002 Cuộc thi Lập trình Sinh viên CTU crit-III-1 8.0000            0.8000  org-khoa-001        False
evt-003         Seminar Khoa học Dữ liệu crit-III-1 4.0000            0.4000  org-khoa-001        False
evt-004             Workshop Kỹ năng Mềm crit-III-1 3.0000            0.3000  org-ttht-001        False
evt-005     Ngày hội Học thuật Khoa CNTT crit-III-1 5.0000            0.5000  org-khoa-001        False
evt-006     Liên hoan Văn nghệ Sinh viên crit-III-2 5.0000            1.0000   org-btc-001        False
evt-007          Giải Cầu Lông Sinh viên crit-III-2 4.0000            0.8000   org-btc-001        False
evt-008          Đêm nhạc CTU Got Talent crit-III-2 6.0000            1.2000 org-hoisy-001        False


In [11]:
# ============================================================
# Sinh viên (15 người)
# ============================================================
student_ids = [f'sv-{i:03d}' for i in range(1, 16)]
df_students = pd.DataFrame({
    'studentId'  : student_ids,
    'fullName'   : [f'Sinh Viên {i}' for i in range(1, 16)],
    'studentCode': [f'B{2100000+i}' for i in range(1, 16)],
})

# ============================================================
# event_registrations (EventRegistration entity)
# studentId, eventId, status: REGISTERED | ATTENDED | ABSENT | CANCELLED
# registeredAt, attendedAt, cancelledAt
# ============================================================
past_event_ids = df_events[~df_events['is_upcoming']]['eventId'].tolist()
reg_data = []
for sid in student_ids:
    joined = random.sample(past_event_ids, k=random.randint(3, min(8, len(past_event_ids))))
    for eid in joined:
        status = random.choices(
            ['ATTENDED', 'REGISTERED', 'ABSENT', 'CANCELLED'],
            weights=[0.60, 0.15, 0.15, 0.10]
        )[0]
        reg_data.append({
            'studentId'  : sid,
            'eventId'    : eid,
            'status'     : status,
            'registeredAt': now - timedelta(days=random.randint(10, 80)),
            'attendedAt' : now - timedelta(days=random.randint(1, 10)) if status == 'ATTENDED' else None,
            'cancelledAt': now - timedelta(days=random.randint(1, 5))  if status == 'CANCELLED' else None,
        })
df_registrations = pd.DataFrame(reg_data).drop_duplicates(['studentId', 'eventId'])

# ============================================================
# student_scores (StudentScore entity)
# studentId, eventId, criteriaId, scoreValue — 1 record mỗi lần cộng điểm
# Tổng điểm = SUM(scoreValue) GROUP BY studentId, criteriaId
# ============================================================
score_data = []
for _, reg in df_registrations[df_registrations['status']=='ATTENDED'].iterrows():
    evt = df_events[df_events['eventId']==reg['eventId']].iloc[0]
    score_data.append({
        'studentId'  : reg['studentId'],
        'eventId'    : reg['eventId'],
        'criteriaId' : evt['criteriaId'],
        'scoreValue' : float(evt['score']),
        'semesterId' : 'sem-2025-hk1',
        'createdAt'  : reg['attendedAt'],
    })
df_student_scores = pd.DataFrame(score_data)

# Tổng điểm theo (studentId, criteriaId) — dùng trong tính deficit
df_score_agg = df_student_scores.groupby(['studentId','criteriaId'])['scoreValue'].sum().reset_index()
df_score_agg.columns = ['studentId', 'criteriaId', 'totalScore']

# ============================================================
# subscriptions + subscription_categories + subscription_criteria
# Subscription entity: studentId
# join tables: subscription_categories (subscriptionId, categoryId)
#              subscription_criteria   (subscriptionId, criteriaId)
# ============================================================
subs_data = []
for sid in student_ids:
    if random.random() < 0.7:
        subs_data.append({
            'studentId'            : sid,
            'subscribed_categories': random.sample(all_cat_ids, k=random.randint(1, 3)),
            'subscribed_criteria'  : random.sample(leaf_criteria_ids, k=random.randint(1, 2)),
        })
df_subscriptions = pd.DataFrame(subs_data)

print(f'👤 Sinh viên   : {len(df_students)}')
print(f'📝 Đăng ký     : {len(df_registrations)} ({(df_registrations["status"]=="ATTENDED").sum()} ATTENDED)')
print(f'🎯 Score records: {len(df_student_scores)} (tổng = SUM mỗi lần cộng điểm)')
print(f'🔔 Subscription: {len(df_subscriptions)}')
print()
print('Điểm tích lũy mẫu (sv-001):')
sv_sample = df_score_agg[df_score_agg['studentId']=='sv-001'].merge(
    df_criterias[['criteriaId','criteriaName','maxScore']], on='criteriaId'
)
print(sv_sample[['criteriaName','totalScore','maxScore']].to_string(index=False))

👤 Sinh viên   : 15
📝 Đăng ký     : 81 (51 ATTENDED)
🎯 Score records: 51 (tổng = SUM mỗi lần cộng điểm)
🔔 Subscription: 11

Điểm tích lũy mẫu (sv-001):
         criteriaName  totalScore  maxScore
  Hoạt động học thuật      9.0000        10
 Hoạt động văn thể mỹ      7.0000         5
Hoạt động tình nguyện     10.0000         5


---
## Cell 3: User Profile Vector

Theo spec, User Profile gồm **6 thành phần**:
```
User Profile = [
  deficit_by_criteria[],     # maxScore - SUM(scoreValue) theo criteriaId
  attended_categories[],     # Tần suất danh mục đã tham dự (implicit)
  attended_organizers[],     # Ban tổ chức quen thuộc (implicit)
  avg_score_preference,      # Điểm sự kiện ưa thích trung bình
  subscribed_categories[],   # Explicit: subscription_categories
  subscribed_criteria[],     # Explicit: subscription_criteria
]
```

In [12]:
def compute_user_profile(student_id: str) -> dict:
    """
    Tính User Profile Vector theo đúng spec recommendation_system_analysis.md.
    Mapping với DB thật:
      - deficit: từ student_scores (SUM scoreValue GROUP BY criteriaId)
      - attended_categories, attended_organizers, avg_score_preference: từ event_registrations JOIN events
      - subscribed_*: từ subscriptions JOIN subscription_categories/criteria
    """
    profile = {'studentId': student_id}

    # ── 1. Deficit: điểm còn thiếu theo từng tiêu chí lá ──────────────
    deficit = {}
    deficit_normalized = {}
    for _, crit_row in df_leaf_criteria.iterrows():
        cid   = crit_row['criteriaId']
        max_s = float(crit_row['maxScore'])
        agg   = df_score_agg[(df_score_agg['studentId']==student_id) & (df_score_agg['criteriaId']==cid)]
        current = float(agg['totalScore'].values[0]) if len(agg) > 0 else 0.0
        current = min(current, max_s)  # Cap tại maxScore (logic của backend)
        deficit[cid]            = max(0.0, max_s - current)
        deficit_normalized[cid] = deficit[cid] / max_s
    profile['deficit']            = deficit
    profile['deficit_normalized'] = deficit_normalized

    # ── 2. Attended events (ATTENDED status only) ─────────────────────
    attended_rows = df_registrations[
        (df_registrations['studentId']==student_id) &
        (df_registrations['status']=='ATTENDED')
    ].merge(df_events[['eventId','categoryIds','organizerId','score']], on='eventId', how='left')

    # ── 3. attended_categories: tần suất danh mục (implicit preference) ──
    cat_counts = {cid: 0 for cid in all_cat_ids}
    for _, row in attended_rows.iterrows():
        for cat in (row['categoryIds'] or []):
            if cat in cat_counts:
                cat_counts[cat] += 1
    total_cat = sum(cat_counts.values())
    profile['attended_categories'] = {
        k: v / total_cat if total_cat > 0 else 0.0
        for k, v in cat_counts.items()
    }

    # ── 4. attended_organizers: tần suất ban tổ chức (implicit preference) ──
    org_counts = {oid: 0 for oid in organizer_ids}
    for _, row in attended_rows.iterrows():
        oid = row.get('organizerId')
        if oid and oid in org_counts:
            org_counts[oid] += 1
    total_org = sum(org_counts.values())
    profile['attended_organizers'] = {
        k: v / total_org if total_org > 0 else 0.0
        for k, v in org_counts.items()
    }

    # ── 5. avg_score_preference: trung bình điểm sự kiện đã tham dự ──
    scores_attended = attended_rows['score'].dropna().tolist()
    profile['avg_score_preference'] = float(np.mean(scores_attended)) if scores_attended else None

    # ── 6. Explicit: subscribed_categories & subscribed_criteria ─────
    sub = df_subscriptions[df_subscriptions['studentId']==student_id]
    profile['subscribed_categories'] = sub.iloc[0]['subscribed_categories'] if len(sub)>0 else []
    profile['subscribed_criteria']   = sub.iloc[0]['subscribed_criteria']   if len(sub)>0 else []

    # ── 7. Metadata ──────────────────────────────────────────────────
    profile['registered_events'] = df_registrations[
        df_registrations['studentId']==student_id
    ]['eventId'].tolist()
    profile['is_new_user'] = len(attended_rows) == 0

    return profile


# Test
p = compute_user_profile('sv-001')
print(f'=== User Profile: sv-001 ===')
print(f'is_new_user        : {p["is_new_user"]}')
print(f'avg_score_preference: {p["avg_score_preference"]}')
print(f'Subscription cats  : {p["subscribed_categories"]}')
print(f'Subscription crits : {p["subscribed_criteria"]}')
print('Deficit (điểm thiếu):')
for cid, val in p['deficit'].items():
    max_s = df_criterias[df_criterias['criteriaId']==cid]['maxScore'].values[0]
    cname = df_criterias[df_criterias['criteriaId']==cid]['criteriaName'].values[0]
    bar = '█' * int(p['deficit_normalized'][cid] * 10)
    print(f'  {cid} ({cname}): thiếu {val}/{max_s} {bar}')
print('attended_organizers:')
for oid, freq in p['attended_organizers'].items():
    if freq > 0:
        oname = df_organizers[df_organizers['organizerId']==oid]['organizerName'].values[0]
        print(f'  {oname}: {freq:.2f}')

=== User Profile: sv-001 ===
is_new_user        : False
avg_score_preference: 5.2
Subscription cats  : ['cat-the-thao']
Subscription crits : ['crit-III-1', 'crit-V-2']
Deficit (điểm thiếu):
  crit-III-1 (Hoạt động học thuật): thiếu 1.0/10 █
  crit-III-2 (Hoạt động văn thể mỹ): thiếu 0.0/5 
  crit-III-3 (Hoạt động tình nguyện): thiếu 0.0/5 
  crit-V-1 (Công tác Đoàn - Hội): thiếu 5.0/5 ██████████
  crit-V-2 (Công tác Đội - nhóm): thiếu 5.0/5 ██████████
attended_organizers:
  Ban Tổ Chức Trường: 0.40
  Đoàn Thanh Niên CTU: 0.20
  Khoa CNTT: 0.40


---
## Cell 4: Xây Dựng CBF Scoring

Trọng số đúng theo spec (`w1=0.5, w2=0.25, w3=0.10, w4=0.10`, tổng = 0.95 như trong analysis.md).

`preference_sim` mở rộng để tích hợp `attended_organizers` và `avg_score_preference`:
```
implicit_sim = 0.6 × cat_sim + 0.25 × organizer_sim + 0.15 × score_sim
preference_sim = α × implicit_sim + β × subscription_match
```

In [8]:
# ── Trọng số CBF (đúng theo spec) ──
W1 = 0.50   # deficit_score
W2 = 0.25   # preference_sim
W3 = 0.10   # not_attended
W4 = 0.10   # upcoming_bonus
# Tổng = 0.95 (theo analysis.md)

# ── Sub-trọng số của implicit preference ──
W_CAT  = 0.60   # category similarity
W_ORG  = 0.25   # organizer familiarity
W_SCORE = 0.15  # score preference match

# ── α/β cho preference_sim ──
ALPHA_HAS_HISTORY = 0.6   # trọng số implicit khi sinh viên có lịch sử
BETA_HAS_HISTORY  = 0.4   # trọng số explicit (subscription)
ALPHA_NEW_USER    = 0.0   # sinh viên mới: không dùng implicit
BETA_NEW_USER     = 1.0   # sinh viên mới: 100% subscription


def score_deficit(profile: dict, event: pd.Series) -> float:
    """w1 component: điểm ưu tiên tiêu chí còn thiếu. [0, 1]"""
    return profile['deficit_normalized'].get(event['criteriaId'], 0.0)


def score_preference_sim(profile: dict, event: pd.Series) -> float:
    """
    w2 component: preference_sim = α × implicit_sim + β × subscription_match

    implicit_sim kết hợp:
      - Category similarity (cosine: attended_cats vs event.cats)
      - Organizer familiarity (attended_organizers)
      - Score preference match (avg_score_preference vs event.score_normalized)
    """
    is_new = profile['is_new_user']
    alpha  = ALPHA_NEW_USER if is_new else ALPHA_HAS_HISTORY
    beta   = BETA_NEW_USER  if is_new else BETA_HAS_HISTORY

    # -- Category cosine similarity --
    att_cat_vec = np.array([profile['attended_categories'].get(c, 0) for c in all_cat_ids])
    evt_cat_vec = np.array([1.0 if c in (event['categoryIds'] or []) else 0.0 for c in all_cat_ids])
    if np.linalg.norm(att_cat_vec) > 0 and np.linalg.norm(evt_cat_vec) > 0:
        cat_sim = float(cosine_similarity([att_cat_vec], [evt_cat_vec])[0][0])
    else:
        cat_sim = 0.0

    # -- Organizer familiarity --
    org_sim = profile['attended_organizers'].get(event['organizerId'], 0.0)

    # -- Score preference match --
    avg_pref = profile['avg_score_preference']
    if avg_pref is not None:
        # Chuẩn hóa avg_pref về [0,1] so sánh với score_normalized của event
        max_possible = max(df_events['score'].max(), 1)
        avg_pref_norm = avg_pref / max_possible
        score_match = 1.0 - abs(event['score_normalized'] - avg_pref_norm)
    else:
        score_match = 0.0

    implicit_sim = W_CAT * cat_sim + W_ORG * org_sim + W_SCORE * score_match

    # -- Subscription match (explicit) --
    sub_cat_match  = any(c in profile['subscribed_categories'] for c in (event['categoryIds'] or []))
    sub_crit_match = event['criteriaId'] in profile['subscribed_criteria']
    subscription_match = 1.0 if (sub_cat_match or sub_crit_match) else 0.0

    return alpha * implicit_sim + beta * subscription_match


def score_not_attended(profile: dict, event: pd.Series) -> float:
    """w3 component: 1.0 nếu chưa đăng ký sự kiện này."""
    return 0.0 if event['eventId'] in profile['registered_events'] else 1.0


def score_upcoming(event: pd.Series) -> float:
    """w4 component: 1.0 nếu sự kiện sắp diễn ra."""
    return 1.0 if event['is_upcoming'] else 0.0


def compute_cbf_score(profile: dict, event: pd.Series) -> dict:
    """Tính toàn bộ CBF score cho một cặp (sinh viên, sự kiện)."""
    s1 = score_deficit(profile, event)
    s2 = score_preference_sim(profile, event)
    s3 = score_not_attended(profile, event)
    s4 = score_upcoming(event)
    total = W1*s1 + W2*s2 + W3*s3 + W4*s4  # tổng max = 0.95 theo spec
    return {
        'eventId'        : event['eventId'],
        'eventName'      : event['eventName'],
        'criteriaId'     : event['criteriaId'],
        'score'          : event['score'],
        'organizerId'    : event['organizerId'],
        'is_upcoming'    : event['is_upcoming'],
        's1_deficit'     : round(s1, 4),
        's2_preference'  : round(s2, 4),
        's3_not_attended': round(s3, 4),
        's4_upcoming'    : round(s4, 4),
        'cbf_score'      : round(total, 4),
    }


print('✅ CBF scoring functions đã định nghĩa!')
print(f'   w1={W1} (deficit), w2={W2} (pref_sim), w3={W3} (not_attended), w4={W4} (upcoming)')
print(f'   Tổng trọng số = {W1+W2+W3+W4} (theo spec analysis.md)')
print(f'   implicit_sim = {W_CAT}×cat_sim + {W_ORG}×org_sim + {W_SCORE}×score_match')
print(f'   α/β có lịch sử: {ALPHA_HAS_HISTORY}/{BETA_HAS_HISTORY}')
print(f'   α/β sinh viên mới: {ALPHA_NEW_USER}/{BETA_NEW_USER}')

✅ CBF scoring functions đã định nghĩa!
   w1=0.5 (deficit), w2=0.25 (pref_sim), w3=0.1 (not_attended), w4=0.1 (upcoming)
   Tổng trọng số = 0.95 (theo spec analysis.md)
   implicit_sim = 0.6×cat_sim + 0.25×org_sim + 0.15×score_match
   α/β có lịch sử: 0.6/0.4
   α/β sinh viên mới: 0.0/1.0


---
## Cell 5: Hàm Gợi Ý & Test

In [9]:
def recommend_cbf(student_id: str, top_n: int = 5, only_upcoming: bool = True) -> pd.DataFrame:
    """
    Gợi ý sự kiện bằng Content-Based Filtering.

    Args:
        student_id   : studentId (uuid)
        top_n        : Số gợi ý
        only_upcoming: Chỉ gợi ý sự kiện sắp tới
    Returns:
        DataFrame sắp xếp theo cbf_score giảm dần
    """
    profile = compute_user_profile(student_id)
    pool = df_events[df_events['is_upcoming']] if only_upcoming else df_events
    results = [compute_cbf_score(profile, row) for _, row in pool.iterrows()]
    return pd.DataFrame(results).sort_values('cbf_score', ascending=False).head(top_n).reset_index(drop=True)


# ── Test SV có lịch sử ──
sid = 'sv-001'
profile = compute_user_profile(sid)
print(f'=== GỢI Ý CBF CHO {sid} ===')
print(f'is_new_user: {profile["is_new_user"]}')
print(f'avg_score_preference: {profile["avg_score_preference"]}')
print()
recs = recommend_cbf(sid, top_n=5)
print('🎯 Top 5 gợi ý:')
print(recs[['eventName','criteriaId','score','s1_deficit','s2_preference','cbf_score']].to_string(index=False))

=== GỢI Ý CBF CHO sv-001 ===
is_new_user: False
avg_score_preference: 7.5

🎯 Top 5 gợi ý:
                    eventName criteriaId  score  s1_deficit  s2_preference  cbf_score
        Ngày hội việc làm CTU   crit-V-2 4.0000      1.0000         0.2325     0.7581
 Chào tân sinh viên 2025-2026   crit-V-2 5.0000      1.0000         0.2145     0.7536
        Kết nạp Đoàn viên mới   crit-V-1 5.0000      1.0000         0.1425     0.7356
       Đại hội Đoàn Khoa CNTT   crit-V-1 6.0000      1.0000         0.1245     0.7311
Hội nghị Ban Chấp hành Hội SV   crit-V-1 4.0000      1.0000         0.0855     0.7214


In [ ]:
# ── Test sinh viên mới (cold-start) ──
NEW_SID = 'sv-new-001'
# Sinh viên mới: chưa có event_registrations, chưa có student_scores
# Chỉ có subscription (đăng ký quan tâm ngay khi tạo tài khoản)
df_subscriptions = pd.concat([df_subscriptions, pd.DataFrame([{
    'studentId'            : NEW_SID,
    'subscribed_categories': ['cat-hoc-thuat', 'cat-ky-nang'],
    'subscribed_criteria'  : ['crit-III-1'],
}])], ignore_index=True)

# Không có student_scores → df_score_agg không có sv này → deficit = maxScore cho mỗi tiêu chí
profile_new = compute_user_profile(NEW_SID)

print(f'=== COLD-START: {NEW_SID} (sinh viên mới) ===')
print(f'is_new_user   : {profile_new["is_new_user"]}')
print(f'α/β dùng      : {ALPHA_NEW_USER}/{BETA_NEW_USER} (100% subscription)')
print(f'Subscription  : cats={profile_new["subscribed_categories"]}')
print()
recs_new = recommend_cbf(NEW_SID, top_n=5)
print('🎯 Top 5 gợi ý (dựa vào subscription + deficit):')
print(recs_new[['eventName','criteriaId','score','s1_deficit','s2_preference','cbf_score']].to_string(index=False))

In [9]:
# ── So sánh gợi ý nhiều sinh viên ──
print('=== SO SÁNH GỢI Ý 5 SINH VIÊN ===')
for sid in ['sv-001', 'sv-002', 'sv-003', 'sv-004', 'sv-005']:
    pf = compute_user_profile(sid)
    top_cid = max(pf['deficit'], key=pf['deficit'].get)
    top_cname = df_criterias[df_criterias['criteriaId']==top_cid]['criteriaName'].values[0]
    recs = recommend_cbf(sid, top_n=3)
    print(f'\n┌─ {sid} (mới: {pf["is_new_user"]}, avg_score_pref: {pf["avg_score_preference"]})')
    print(f'│ Tiêu chí thiếu nhất: {top_cid} ({top_cname})')
    print(f'│ Subscription: {pf["subscribed_categories"]}')
    for _, row in recs.iterrows():
        print(f'│ → [{row["cbf_score"]:.4f}] {row["eventName"]} ({row["criteriaId"]}, {row["score"]} điểm)')

=== SO SÁNH GỢI Ý 5 SINH VIÊN ===

┌─ sv-001 (mới: False, avg_score_pref: 6.0)
│ Tiêu chí thiếu nhất: crit-III-1 (Hoạt động học thuật)
│ Subscription: ['cat-van-hoa', 'cat-tinh-nguyen']
│ → [0.8597] Chào tân sinh viên 2025-2026 (crit-V-2, 5.0 điểm)
│ → [0.8392] Ngày hội việc làm CTU (crit-V-2, 4.0 điểm)
│ → [0.7260] Kết nạp Đoàn viên mới (crit-V-1, 5.0 điểm)

┌─ sv-002 (mới: False, avg_score_pref: 7.666666666666667)
│ Tiêu chí thiếu nhất: crit-III-1 (Hoạt động học thuật)
│ Subscription: ['cat-van-hoa', 'cat-doan-hoi', 'cat-hoc-thuat']
│ → [0.8737] Ngày hội việc làm CTU (crit-V-2, 4.0 điểm)
│ → [0.8692] Chào tân sinh viên 2025-2026 (crit-V-2, 5.0 điểm)
│ → [0.8422] Kết nạp Đoàn viên mới (crit-V-1, 5.0 điểm)

┌─ sv-003 (mới: False, avg_score_pref: 6.2)
│ Tiêu chí thiếu nhất: crit-III-1 (Hoạt động học thuật)
│ Subscription: ['cat-the-thao']
│ → [0.7744] Chào tân sinh viên 2025-2026 (crit-V-2, 5.0 điểm)
│ → [0.7714] Ngày hội việc làm CTU (crit-V-2, 4.0 điểm)
│ → [0.7290] Kết nạp Đoàn viên 

---
## Cell 6: Tổng Kết CBF

**Ưu điểm:**
- ✅ Không cần dữ liệu sinh viên khác (không phụ thuộc cộng đồng)
- ✅ Cold-start: sinh viên mới → dùng subscription (α=0, β=1)
- ✅ Triển khai được ngay từ ngày đầu
- ✅ Đầy đủ 6 thành phần User Profile theo spec

**Nhược điểm:**
- ⚠️ Filter bubble: chỉ gợi ý những gì đã biết
- ⚠️ Không học từ hành vi cộng đồng

**Bước tiếp theo:** Xem `02_collaborative_filtering.ipynb` và `03_hybrid_filtering.ipynb`

In [10]:
print('📊 Tổng kết Content-Based Filtering')
print('=' * 55)
print(f'Sinh viên  : {len(student_ids)}')
print(f'Sự kiện    : {len(df_events)} ({df_events["is_upcoming"].sum()} sắp tới)')
print(f'Score records (student_scores): {len(df_student_scores)}')
print(f'Trọng số   : w1={W1}, w2={W2}, w3={W3}, w4={W4} (tổng={W1+W2+W3+W4})')
print()
print('Phân phối cbf_score trên 10 sv × tất cả sự kiện sắp tới:')
all_scores = []
for sid in student_ids[:10]:
    pf = compute_user_profile(sid)
    for _, evt in df_events[df_events['is_upcoming']].iterrows():
        all_scores.append(compute_cbf_score(pf, evt)['cbf_score'])
arr = np.array(all_scores)
print(f'  Min : {arr.min():.4f}')
print(f'  Max : {arr.max():.4f}')
print(f'  Mean: {arr.mean():.4f}')
print(f'  Std : {arr.std():.4f}')

📊 Tổng kết Content-Based Filtering
Sinh viên  : 15
Sự kiện    : 20 (6 sắp tới)
Score records (student_scores): 49
Trọng số   : w1=0.5, w2=0.25, w3=0.1, w4=0.1 (tổng=0.95)

Phân phối cbf_score trên 10 sv × tất cả sự kiện sắp tới:
  Min : 0.2479
  Max : 0.8737
  Mean: 0.7004
  Std : 0.1719


---
## Cell 7: Chuẩn Hóa CBF Score (Min-Max) Cho Hybrid

Để kết hợp với CF trong Hybrid, chuẩn hóa điểm CBF về cùng thang đo:
```
x_norm = (x - min) / (max - min)
```

Notebook này vẫn giữ `cbf_score` gốc để giải thích; thêm `cbf_norm` cho bước trộn hybrid.

In [10]:
def minmax_normalize(scores: dict) -> dict:
    """Min-Max normalization: x_norm = (x - min) / (max - min)."""
    vals = [v for v in scores.values() if v is not None and not np.isnan(v)]
    if not vals or max(vals) == min(vals):
        return {k: (np.nan if (v is None or np.isnan(v)) else 0.5) for k, v in scores.items()}
    lo, hi = min(vals), max(vals)
    return {k: (np.nan if (v is None or np.isnan(v)) else (v - lo) / (hi - lo)) for k, v in scores.items()}


def recommend_cbf_with_norm(student_id: str, top_n: int = 5, only_upcoming: bool = True) -> pd.DataFrame:
    """
    Trả về cả điểm gốc và điểm chuẩn hóa để dùng cho Hybrid.
    - cbf_score: điểm CBF gốc
    - cbf_norm : điểm CBF sau Min-Max
    """
    raw_df = recommend_cbf(student_id=student_id, top_n=1000, only_upcoming=only_upcoming)
    raw_scores = {row['eventId']: float(row['cbf_score']) for _, row in raw_df.iterrows()}
    norm_scores = minmax_normalize(raw_scores)
    raw_df['cbf_norm'] = raw_df['eventId'].map(norm_scores).astype(float)
    return raw_df.sort_values('cbf_norm', ascending=False).head(top_n).reset_index(drop=True)


print('✅ Đã thêm chuẩn hóa CBF theo Min-Max cho Hybrid')
print('   Công thức: x_norm = (x - min) / (max - min)')
print()
demo_cbf_norm = recommend_cbf_with_norm('sv-001', top_n=5)
print(demo_cbf_norm[['eventName', 'cbf_score', 'cbf_norm']].to_string(index=False))

✅ Đã thêm chuẩn hóa CBF theo Min-Max cho Hybrid
   Công thức: x_norm = (x - min) / (max - min)

                    eventName  cbf_score  cbf_norm
        Ngày hội việc làm CTU     0.7581    1.0000
 Chào tân sinh viên 2025-2026     0.7536    0.9910
        Kết nạp Đoàn viên mới     0.7356    0.9548
       Đại hội Đoàn Khoa CNTT     0.7311    0.9458
Hội nghị Ban Chấp hành Hội SV     0.7214    0.9263
